# Снигур (ЦФТ) и моя выгрузка из Альфы: быстрый итог

- У Снигура берётся весь файл, как есть.
- Моя выгрузка - лист `С ТЭ на 01.09.2026` из `trade_acquiring_clients_2026-09-01.xlsx`.
- К озеру один запрос: `ods_alpha.scd1_agreements` + `scd1_companies` (около минуты). Из него ИНН и наименование для номеров Снигура и причины.
- `scd1_agr_terms` не читается. Если договор Снигура в Альфе проходит все фильтры, кроме привязки P, а в моей выгрузке его нет, причина - привязка терминала P.
- Клиент - ИНН, если его нет - наименование без ОПФ и знаков препинания.


In [ ]:
import getpass
import re
import time
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

AS_OF_DATE = '2026-09-01'
BASE_DIR = Path('/home/jovyan/documents/Equaring')
SNIGUR_FILE = BASE_DIR / 'Скрипт для Ани по сборке продукта эквайринг' / 'cdwh_acq_clients.xlsx'
SNIGUR_SHEET = 0
MINE_FILE = BASE_DIR / ('trade_acquiring_clients_%s.xlsx' % AS_OF_DATE)
OUT_FILE = BASE_DIR / ('te_snigur_vs_mine_quick_%s.xlsx' % AS_OF_DATE)

as_of = pd.Timestamp(AS_OF_DATE)
as_of_ru = as_of.strftime('%d.%m.%Y')
MINE_SHEET = 'С ТЭ на %s' % as_of_ru

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()
print('imp ready')


## 1. Файлы

In [ ]:
def norm_agr(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r'\d+\.0', s):
        s = s[:-2]
    s = re.sub(r'\s+', '', s).upper()
    return None if s in ('', 'NONE', 'NAN', '<NA>') else s

def to_date(s):
    s = pd.Series(s, dtype=object)
    iso = s.map(lambda v: isinstance(v, str) and bool(re.match(r'\s*\d{4}-\d{2}-\d{2}', v)))
    out = pd.to_datetime(s.where(~iso), errors='coerce', dayfirst=True)
    out[iso] = pd.to_datetime(s[iso].str.strip().str[:10], errors='coerce', format='%Y-%m-%d')
    return out.dt.normalize()

def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()

def find_col(df, *keys, exact=None):
    if exact and exact in df.columns:
        return exact
    hits = [c for c in df.columns if all(k in _low(c) for k in keys)]
    if len(hits) == 1:
        return hits[0]
    raise KeyError('Колонка %s: найдено %s. Все колонки: %s' % (keys, hits, list(df.columns)))

raw = pd.read_excel(SNIGUR_FILE, sheet_name=SNIGUR_SHEET, header=None, dtype=object)
hdr = next(i for i in range(min(20, len(raw))) if any('договор' in _low(v) for v in raw.iloc[i] if pd.notna(v)))
sn_raw = raw.iloc[hdr + 1:].copy()
sn_raw.columns = [_low(v) if pd.notna(v) else 'col_%s' % j for j, v in enumerate(raw.iloc[hdr])]
sn_raw = sn_raw.dropna(how='all')

c_agr = find_col(sn_raw, 'номер', 'договор', exact='номер договора')
c_end = find_col(sn_raw, 'окончан', exact='дата окончания действия договора')
sn = pd.DataFrame({
    'agr_key': sn_raw[c_agr].map(norm_agr),
    'Номер договора (Снигур)': sn_raw[c_agr].astype('string').str.strip(),
    'Клиент (Снигур)': sn_raw[find_col(sn_raw, 'наимен', exact='наименование клиента')].astype('string').str.strip(),
    'Статус (Снигур)': sn_raw[find_col(sn_raw, 'статус', exact='статус')].astype('string').str.strip(),
    'Дата окончания (Снигур)': to_date(sn_raw[c_end]),
    'Дата закрытия (Снигур)': to_date(sn_raw[find_col(sn_raw, 'закрыт', 'дата', exact='дата закрытия')]),
})
sn_rows = len(sn)
sn = sn[sn['agr_key'].notna()].drop_duplicates('agr_key').reset_index(drop=True)

mine_raw = pd.read_excel(MINE_FILE, sheet_name=MINE_SHEET, dtype=object)
my = pd.DataFrame({
    'agr_key': mine_raw['Номер договора'].map(norm_agr),
    'Номер договора (мой)': mine_raw['Номер договора'].astype('string'),
    'ИНН (мой)': mine_raw['ИНН'].astype('string'),
    'ЦФТ ID': mine_raw['ЦФТ ID'].astype('string'),
})
my = my[my['agr_key'].notna()].drop_duplicates('agr_key').reset_index(drop=True)
print('Снигур: строк %s, уникальных номеров %s | у меня: %s' % (sn_rows, len(sn), len(my)))


## 2. Один запрос к озеру: договоры и клиенты

In [ ]:
t0 = time.time()
print('Запрос пошёл в', time.strftime('%H:%M:%S'), flush=True)
with imp:
    imp.execute('set MEM_LIMIT=16g')
    lake = imp.fetch('''
SELECT
  upper(regexp_replace(a.c_agr_number, '[[:space:]]', '')) AS agr_key,
  cast(a.n_agr AS string) AS n_agr,
  a.acq_class,
  a.d_valid_from,
  a.d_valid_to,
  coalesce(a.ods_deleted_flg, '0') AS agr_deleted,
  regexp_replace(c.c_inn, '[^0-9]', '') AS inn,
  c.c_cmp_name AS cmp_name,
  coalesce(c.ods_deleted_flg, '0') AS cmp_deleted
FROM ods_alpha.scd1_agreements a
LEFT JOIN ods_alpha.scd1_companies c ON c.n_cmp = a.n_cmp_client
WHERE a.c_agr_number IS NOT NULL
''')
lake.columns = [str(c).split('.')[-1].lower() for c in lake.columns]
lake['agr_deleted'] = lake['agr_deleted'].astype(str)
lake['cmp_deleted'] = lake['cmp_deleted'].astype(str)
lake['d_valid_from'] = pd.to_datetime(lake['d_valid_from'], errors='coerce').dt.normalize()
lake['d_valid_to'] = pd.to_datetime(lake['d_valid_to'], errors='coerce').dt.normalize()
print('Строк: %s за %.0f с' % (len(lake), time.time() - t0))


## 3. Итог

In [ ]:
LEGAL_RE = re.compile(
    r'\b(ООО|ОАО|ЗАО|ПАО|АО|НАО|ИП|КФХ|ГКФХ|СПК|СХПК|ТОО|ПК|ООО КФХ|'
    r'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ|ИНДИВИДУАЛЬНЫЙ ПРЕДПРИНИМАТЕЛЬ|'
    r'АКЦИОНЕРНОЕ ОБЩЕСТВО|КРЕСТЬЯНСКОЕ \(?ФЕРМЕРСКОЕ\)? ХОЗЯЙСТВО|ГЛАВА)\b')

def norm_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D', '', str(v))
    if len(s) in (9, 11):
        s = '0' + s
    return s or None

def norm_name(v):
    if pd.isna(v) or not str(v).strip():
        return None
    s = str(v).upper().replace('Ё', 'Е')
    s = re.sub(r'[^\w\s]', ' ', s)
    s = LEGAL_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s or None

cmp_ok = lake[lake['cmp_deleted'].ne('1')]
inn_by_key = cmp_ok[cmp_ok['inn'].fillna('').ne('')].drop_duplicates('agr_key').set_index('agr_key')['inn']
name_by_key = cmp_ok.dropna(subset=['cmp_name']).drop_duplicates('agr_key').set_index('agr_key')['cmp_name']

sn['inn'] = sn['agr_key'].map(inn_by_key).map(norm_inn)
sn['name_n'] = sn['Клиент (Снигур)'].map(norm_name)
my['inn'] = my['ИНН (мой)'].map(norm_inn)
my['name_n'] = my['agr_key'].map(name_by_key).map(norm_name)
for df in (sn, my):
    df['client'] = df['inn'].where(df['inn'].notna(), 'Н:' + df['name_n'].fillna('?'))

both_keys = set(sn['agr_key']) & set(my['agr_key'])
sn['Есть у меня'] = sn['agr_key'].isin(both_keys)
my['Есть у Снигура'] = my['agr_key'].isin(both_keys)

my_inns, my_names = set(my['inn'].dropna()), set(my['name_n'].dropna())
sn_inns, sn_names = set(sn['inn'].dropna()), set(sn['name_n'].dropna())

def found_how(inn, name, inns, names):
    if inn and inn in inns:
        return 'по ИНН'
    if name and name in names:
        return 'по наименованию'
    return None

sn['Клиент у меня'] = [found_how(i, n, my_inns, my_names) for i, n in zip(sn['inn'], sn['name_n'])]
my['Клиент у Снигура'] = [found_how(i, n, sn_inns, sn_names) for i, n in zip(my['inn'], my['name_n'])]

def stage(r):
    if r['agr_deleted'] == '1':
        return 1, 'В Альфе договор удалён'
    if str(r['acq_class'] if pd.notna(r['acq_class']) else '').strip().upper() != 'SA':
        return 2, 'В Альфе класс договора не SA'
    if pd.notna(r['d_valid_from']) and r['d_valid_from'] > as_of:
        return 3, 'В Альфе договор начинается позже %s' % as_of_ru
    if pd.notna(r['d_valid_to']) and r['d_valid_to'] < as_of:
        return 3, 'В Альфе срок закончился до %s' % as_of_ru
    if r['cmp_deleted'] == '1' or pd.isna(r['inn']) or not r['inn']:
        return 4, 'В Альфе нет клиента или ИНН'
    return 5, 'В Альфе остальное в порядке: нет действующей привязки терминала P'

only_sn = sn[~sn['Есть у меня']].copy()
only_my = my[~my['Есть у Снигура']].copy()

cand = lake[lake['agr_key'].isin(set(only_sn['agr_key']))].copy()
if len(cand):
    st = cand.apply(stage, axis=1, result_type='expand')
    cand['stage'], cand['reason'] = st[0], st[1]
    best = cand.sort_values('stage', ascending=False).drop_duplicates('agr_key')
    best = best[['agr_key', 'reason', 'd_valid_to']].rename(columns={'d_valid_to': 'Дата окончания (Альфа)'})
else:
    best = pd.DataFrame(columns=['agr_key', 'reason', 'Дата окончания (Альфа)'])
only_sn = only_sn.merge(best, on='agr_key', how='left')

def sn_reason(r):
    if pd.isna(r['reason']):
        return ('Номера нет в Альфе, клиент есть у меня под другим номером' if r['Клиент у меня']
                else 'Номера нет в Альфе, клиента у меня нет')
    if r['reason'].startswith('В Альфе срок закончился'):
        st = r['Статус (Снигур)'] if pd.notna(r['Статус (Снигур)']) else 'пусто'
        return '%s, у Снигура статус «%s»' % (r['reason'], st)
    return r['reason']

only_sn['Причина'] = only_sn.apply(sn_reason, axis=1) if len(only_sn) else None
only_my['Причина'] = [
    'Тот же клиент у Снигура под другим номером (%s)' % h if h else 'Клиента у Снигура нет ни по ИНН, ни по наименованию'
    for h in only_my['Клиент у Снигура']
]

def client_split(df, col_found):
    per = df.assign(m=df['agr_key'].isin(both_keys) | df[col_found].notna()).groupby('client')['m'].any()
    return len(per), int(per.sum()), int((~per).sum())

sc, mc = client_split(sn, 'Клиент у меня'), client_split(my, 'Клиент у Снигура')
counts = pd.DataFrame([
    ('Договоров', len(sn), len(my)),
    ('  из них с датой окончания раньше %s' % as_of_ru, int((sn['Дата окончания (Снигур)'] < as_of).sum()), None),
    ('Клиентов по ИНН', sn['inn'].nunique(), my['inn'].nunique()),
    ('  договоров без ИНН (номер не найден в Альфе)', int(sn['inn'].isna().sum()), int(my['inn'].isna().sum())),
    ('Клиентов по наименованию', sn['name_n'].nunique(), my['name_n'].nunique()),
    ('Клиентов (ИНН, иначе наименование)', sc[0], mc[0]),
    ('Договоров есть у обоих', len(both_keys), len(both_keys)),
    ('Договоров только у этой стороны', len(only_sn), len(only_my)),
    ('Клиентов есть у другой стороны', sc[1], mc[1]),
    ('Клиентов нет у другой стороны', sc[2], mc[2]),
], columns=['Показатель', 'Снигур (весь файл)', 'Моя выгрузка на %s' % as_of_ru])

def reason_table(df, title):
    t = df['Причина'].value_counts().rename_axis(title).reset_index(name='Договоров')
    t['Доля, %'] = (t['Договоров'] / max(len(df), 1) * 100).round(1)
    return pd.concat([t, pd.DataFrame([{title: 'Всего', 'Договоров': len(df), 'Доля, %': 100.0}])], ignore_index=True)

reasons_sn = reason_table(only_sn, 'Только у Снигура: почему нет у меня')
reasons_my = reason_table(only_my, 'Только у меня: почему нет у Снигура')
calc = len(sn) - len(only_sn) + len(only_my)
math = pd.DataFrame([
    ('Договоров у Снигура', len(sn)),
    ('− только у Снигура', len(only_sn)),
    ('+ только у меня', len(only_my)),
    ('= договоров у меня (расчёт)', calc),
    ('Договоров у меня (факт)', len(my)),
    ('Сходится', 'да' if calc == len(my) else 'нет'),
], columns=['Сверка количества', 'Значение'])

for t in (counts, math, reasons_sn, reasons_my):
    display(t)


## 4. Excel

In [ ]:
def autofit(ws):
    for col in ws.columns:
        width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 10), 70)

def dates(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    return df

sn_cols = ['Номер договора (Снигур)', 'Клиент (Снигур)', 'Статус (Снигур)', 'Дата окончания (Снигур)',
           'Дата закрытия (Снигур)', 'inn', 'Клиент у меня']
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as w:
    row = 0
    for t in (counts, math, reasons_sn, reasons_my):
        t.to_excel(w, index=False, sheet_name='Итог', startrow=row)
        row += len(t) + 2
    dates(only_sn[sn_cols + ['Дата окончания (Альфа)', 'Причина']]).rename(columns={'inn': 'ИНН (Альфа)'}).to_excel(
        w, index=False, sheet_name='Только у Снигура')
    only_my[['Номер договора (мой)', 'ИНН (мой)', 'ЦФТ ID', 'Клиент у Снигура', 'Причина']].assign(
        **{'Наименование (Альфа)': only_my['agr_key'].map(name_by_key)}).to_excel(
        w, index=False, sheet_name='Только у меня')
    dates(sn[sn['Есть у меня']][sn_cols[:5]]).to_excel(w, index=False, sheet_name='Есть у обоих')
    for ws in w.book.worksheets:
        autofit(ws)
        if ws.title != 'Итог':
            ws.freeze_panes = 'A2'
print(OUT_FILE)
